# 🧠 Алгоритми та структури даних · Урок 3 — Дерева, купи та Union-Find

**Дерева** — ієрархічні структури, на яких тримаються бази даних (індекси), файлові системи,
парсери, автодоповнення. Розберемо двійкові дерева й обходи, дерево пошуку (BST), збалансовані
дерева, **купу** (heap) та **Union-Find**.

**Передумова:** [Урок 2 — базові структури](lektsiya-2-bazovi-struktury-danykh.ipynb).

### Що ви вмітимете
- знати **термінологію** дерев і чотири **обходи**;
- реалізувати **BST** (дерево пошуку) і розуміти, коли воно вироджується;
- користуватись **купою**/`heapq` як чергою з пріоритетом;
- розуміти **Trie** та **Union-Find**.

## 1. Термінологія дерев

**Дерево** — вузли, з'єднані ребрами, без циклів; є один **корінь** (root), решта вузлів мають
рівно одного **батька**.

```
            (1)          <- корінь (root)
           /   \
        (2)     (3)      <- (2) — батько (4) і (5)
        /  \       \
     (4)   (5)     (6)   <- листки (leaf) — без дітей: 4, 5, 6
```

- **Корінь** — верхній вузол; **листок (leaf)** — вузол без дітей.
- **Батько/дитина (parent/child)**, **нащадки/предки**.
- **Глибина (depth)** вузла — відстань від кореня; **висота (height)** дерева — найдовший шлях
  від кореня до листка.
- **Піддерево (subtree)** — вузол з усіма його нащадками.

**Двійкове дерево (binary tree)** — у кожного вузла **щонайбільше 2** дитини (left, right).

## 2. Обходи двійкового дерева

Обійти дерево = відвідати всі вузли в певному порядку. Чотири класичні обходи:

- **DFS (у глибину)** — рекурсивно вглиб:
  - **pre-order** (корінь → лівий → правий);
  - **in-order** (лівий → корінь → правий) — для BST дає **відсортований** порядок!
  - **post-order** (лівий → правий → корінь) — напр. для видалення дерева.
- **BFS (у ширину)** — по **рівнях**, зліва направо (через чергу).

In [ ]:
from collections import deque

class TreeNode:
    def __init__(self, value):
        self.value = value
        self.left = None
        self.right = None

#          1
#        /   \
#       2     3
#      / \     \
#     4   5     6
root = TreeNode(1)
root.left = TreeNode(2); root.right = TreeNode(3)
root.left.left = TreeNode(4); root.left.right = TreeNode(5)
root.right.right = TreeNode(6)

def preorder(node, out):
    if node is None: return
    out.append(node.value)          # корінь
    preorder(node.left, out)
    preorder(node.right, out)

def inorder(node, out):
    if node is None: return
    inorder(node.left, out)
    out.append(node.value)          # корінь між піддеревами
    inorder(node.right, out)

def postorder(node, out):
    if node is None: return
    postorder(node.left, out)
    postorder(node.right, out)
    out.append(node.value)          # корінь останнім

def bfs(root):
    out, q = [], deque([root])
    while q:
        node = q.popleft()
        out.append(node.value)
        if node.left: q.append(node.left)
        if node.right: q.append(node.right)
    return out

for name, fn in [("pre-order", preorder), ("in-order", inorder), ("post-order", postorder)]:
    acc = []; fn(root, acc)
    print(f"{name:>11}: {acc}")
print(f"{'BFS (рівні)':>11}: {bfs(root)}")

## 3. Двійкове дерево пошуку (BST)

**BST (Binary Search Tree)** — двійкове дерево з **інваріантом упорядкування**: для кожного вузла
всі значення в **лівому** піддереві **менші**, у **правому** — **більші**.

```
        8
      /   \
     3     10
    / \      \
   1   6      14
```

Завдяки цьому пошук працює як двійковий: на кожному кроці йдемо **ліворуч або праворуч**,
відкидаючи половину — **O(висоти)**. Для **збалансованого** дерева це **O(log n)**.

In [ ]:
class BSTNode:
    def __init__(self, value):
        self.value = value
        self.left = None
        self.right = None

class BST:
    def __init__(self):
        self.root = None

    def insert(self, value):
        self.root = self._insert(self.root, value)

    def _insert(self, node, value):
        if node is None:
            return BSTNode(value)
        if value < node.value:
            node.left = self._insert(node.left, value)      # менше -> ліворуч
        elif value > node.value:
            node.right = self._insert(node.right, value)    # більше -> праворуч
        return node

    def contains(self, value):
        node = self.root
        while node is not None:
            if value == node.value:
                return True
            node = node.left if value < node.value else node.right   # відкидаємо половину
        return False

    def inorder(self):                # in-order обхід BST -> відсортований список
        out = []
        def go(n):
            if n:
                go(n.left); out.append(n.value); go(n.right)
        go(self.root)
        return out

bst = BST()
for x in [8, 3, 10, 1, 6, 14, 4]:
    bst.insert(x)
print("in-order (відсортовано!):", bst.inorder())   # [1, 3, 4, 6, 8, 10, 14]
print("є 6? ", bst.contains(6))
print("є 7? ", bst.contains(7))

> ⚠️ **Проблема виродження.** Якщо вставляти вже впорядковані дані (1, 2, 3, 4, 5), BST
> перетворюється на… звичайний список — висота стає O(n), і пошук деградує до **O(n)**.
>
> ```
> 1
>  \
>   2
>    \
>     3   <- фактично зв'язаний список!
> ```
>
> Тому в проді використовують **самобалансовані** дерева.

## 4. 🚀 Middle+: Збалансовані дерева (AVL, червоно-чорні)

Щоб висота лишалась **O(log n)** за будь-яких вставок, дерево **балансує себе** — після операцій
робить **повороти** (rotations), вирівнюючи піддерева.

- **AVL-дерево** — суворо збалансоване (різниця висот піддерев ≤ 1). Швидший **пошук**, більше
  поворотів при вставці.
- **Червоно-чорне дерево (Red-Black)** — слабше збалансоване, зате менше поворотів. Використовується
  скрізь: `TreeMap`/`TreeSet` у Java, `std::map` у C++, планувальник Linux.
- **B-дерево / B+дерево** — «широкі» дерева для **дисків** і **індексів БД** (див.
  [Модуль 3, Урок 2 — індекси](../modul-3-bazy-danykh/lektsiya-2-bd-z-python-sqlalchemy.ipynb)).

Junior-у досить **розуміти навіщо** (тримати O(log n) і не вироджуватись); реалізовувати повороти
напам'ять — рівень Middle+. Але сам факт «BST може виродитись, тому є самобалансовані дерева»
знати **обов'язково**.

## 5. Купа (heap) та черга з пріоритетом

**Двійкова купа (binary heap)** — майже повне двійкове дерево з інваріантом:
- **min-heap:** батько **≤** дітей (у корені — **мінімум**);
- **max-heap:** батько **≥** дітей (у корені — максимум).

Купу зручно зберігати в **масиві** (без вузлів!): для індексу `i` діти — `2i+1` і `2i+2`, батько —
`(i-1)//2`. Операції **push/pop — O(log n)**, а «підглянути мінімум» — **O(1)**.

**Черга з пріоритетом (priority queue)** — саме її реалізує купа: завжди швидко дістати
найменший/найбільший елемент. У Python — модуль **`heapq`** (min-heap над звичайним списком).

**Де застосовується:** Dijkstra (Урок 5), планувальники задач, top-K елементів, злиття потоків.

In [ ]:
import heapq

# heapq робить min-heap зі звичайного списку
nums = [5, 1, 8, 3, 9, 2]
heapq.heapify(nums)               # перетворити на купу за O(n)
print("корінь (мінімум):", nums[0])

heapq.heappush(nums, 0)           # додати -> O(log n)
print("після push 0, мінімум:", nums[0])

order = [heapq.heappop(nums) for _ in range(3)]   # дістаємо 3 найменші по черзі
print("три найменші поспіль:", order)             # [0, 1, 2]

# Черга з пріоритетом: (пріоритет, задача). Менший пріоритет -> обслуговується першим
pq = []
heapq.heappush(pq, (2, "помити посуд"))
heapq.heappush(pq, (1, "загасити пожежу"))
heapq.heappush(pq, (3, "погуляти"))
print("\nобслуговування за пріоритетом:")
while pq:
    prio, task = heapq.heappop(pq)
    print(f"  [{prio}] {task}")

## 6. Trie (префіксне дерево)

**Trie** (вимовляють «трай», від re**trie**val) — дерево для зберігання **рядків за спільними
префіксами**. Кожне ребро — символ; шлях від кореня до вузла — префікс.

```
        (root)
        /    \
       c      d
       |      |
       a      o
      / \     |
     t   r    g
   [cat][car][dog]
```

Дає **пошук слова / префікса за O(довжини слова)** незалежно від кількості слів. Основа
**автодоповнення**, перевірки орфографії, T9, IP-роутингу.

In [ ]:
class Trie:
    def __init__(self):
        self.children = {}        # символ -> Trie
        self.is_word = False      # чи закінчується тут слово

    def insert(self, word):
        node = self
        for ch in word:
            node = node.children.setdefault(ch, Trie())   # спускаємось/створюємо гілку
        node.is_word = True

    def search(self, word):       # чи є точно таке слово
        node = self._walk(word)
        return node is not None and node.is_word

    def starts_with(self, prefix):  # чи є хоч одне слово з таким префіксом
        return self._walk(prefix) is not None

    def _walk(self, s):
        node = self
        for ch in s:
            if ch not in node.children:
                return None
            node = node.children[ch]
        return node

t = Trie()
for w in ["cat", "car", "dog"]:
    t.insert(w)
print("search('car'):", t.search("car"))          # True
print("search('ca') :", t.search("ca"))           # False (це лише префікс)
print("starts_with('ca'):", t.starts_with("ca"))  # True
print("starts_with('do'):", t.starts_with("do"))  # True
print("starts_with('x') :", t.starts_with("x"))   # False

## 7. Union-Find (система неперетинних множин, DSU)

**Union-Find** (Disjoint Set Union) відповідає на питання «**чи в одній групі** ці два елементи?»
і вміє **об'єднувати** групи. Дві операції:
- **find(x)** — знайти «представника» (корінь) групи елемента;
- **union(a, b)** — об'єднати групи `a` і `b`.

З оптимізаціями **стиснення шляху (path compression)** + **об'єднання за рангом (union by rank)**
кожна операція — практично **O(1)** (точніше — оберненa функція Аккермана, ~стала).

**Де застосовується:** алгоритм Kruskal (MST, Урок 5), пошук компонент зв'язності, «чи з'єднані
два вузли в мережі», угруповання.

In [ ]:
class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))    # спершу кожен сам собі група
        self.rank = [0] * n

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]   # стиснення шляху
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False                 # уже в одній групі
        if self.rank[ra] < self.rank[rb]:  # менше дерево чіпляємо до більшого
            ra, rb = rb, ra
        self.parent[rb] = ra
        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1
        return True

uf = UnionFind(6)          # елементи 0..5
uf.union(0, 1)
uf.union(1, 2)
uf.union(3, 4)
print("0 і 2 разом? ", uf.find(0) == uf.find(2))   # True (0-1-2)
print("0 і 3 разом? ", uf.find(0) == uf.find(3))   # False
uf.union(2, 3)                                      # з'єднали дві групи
print("0 і 4 разом? ", uf.find(0) == uf.find(4))   # True тепер

# ✅ Підсумок уроку
- **Дерево** — ієрархія без циклів; **двійкове** — ≤2 дитини. Обходи: **pre/in/post-order** (DFS) і **BFS** по рівнях.
- **BST** — ліве < вузол < праве; пошук O(висоти); in-order дає відсортований порядок; **вироджується** на впорядкованих даних → O(n).
- **Збалансовані** (AVL, red-black, B-tree) тримають висоту O(log n) через повороти — основа БД-індексів.
- **Купа** — min/max у корені; масив замість вузлів; push/pop O(log n); `heapq` = **черга з пріоритетом**.
- **Trie** — дерево префіксів; пошук слова/префікса за O(довжини); автодоповнення.
- **Union-Find** — «чи в одній групі»; з path compression + rank практично O(1); MST, компоненти.

### ▶️ Далі
Урок 4 — **сортування та пошук**.

### 📚 Хочу знати більше
- Візуалізації дерев/купи: <https://visualgo.net/en/bst>, <https://visualgo.net/en/heap>
- `heapq`: <https://docs.python.org/3/library/heapq.html>
- Union-Find: <https://cp-algorithms.com/data_structures/disjoint_set_union.html>